# MonIA Kaggle Video Studio
Candidate-only GPU worker for Marion & Lucas. Gameplay keeps narrative authority; generated clips never publish automatically. Character identity comes from canonical references. Motion references are secondary and may never provide identity.

In [ ]:
!pip -q install -U diffusers transformers accelerate safetensors imageio[ffmpeg] huggingface_hub requests pillow ftfy

In [ ]:
import os,json,time,requests,torch
from pathlib import Path
from diffusers.utils import export_to_video, load_image, load_video
REPO=os.environ.get('MONIA_REPO','vartcom38-collab/marion-lucas-game')
BRANCH=os.environ.get('MONIA_BRANCH','main')
RAW=f'https://raw.githubusercontent.com/{REPO}/{BRANCH}'
WORK=Path('/kaggle/working/monia-studio'); WORK.mkdir(parents=True,exist_ok=True)
print('MonIA Studio · candidate-only')

In [ ]:
def load_json(url):
    r=requests.get(url,timeout=30); r.raise_for_status(); return r.json()
def download(url,target):
    r=requests.get(url,timeout=120); r.raise_for_status(); Path(target).write_bytes(r.content); return str(target)
def resolve_url(value):
    return RAW+value if value.startswith('/') else value
def validate_job(job):
    assert job.get('candidateOnly') is True
    assert job.get('narrativeAuthority') is False
    assert job.get('source') in {'gameplay','anticipation','review-regeneration'}
    assert (job.get('motion') or {}).get('copyIdentity',False) is False
    assert job.get('output',{}).get('candidatePath')
    return job

## Load one queued job
Set `MONIA_JOB_URL` to a raw GitHub job JSON. One heavy job is processed per run.

In [ ]:
JOB_URL=os.environ.get('MONIA_JOB_URL','').strip()
if not JOB_URL: raise RuntimeError('Set MONIA_JOB_URL')
job=validate_job(load_json(JOB_URL)); job_id=job['id']
job_dir=WORK/job_id; job_dir.mkdir(parents=True,exist_ok=True)
(job_dir/'job.json').write_text(json.dumps(job,ensure_ascii=False,indent=2),encoding='utf-8')
refs={}
for ch in job.get('characters',[]):
    refs[ch['id']]=download(resolve_url(ch['canonRef']),job_dir/f"canon-{ch['id']}.jpg")
previous=(job.get('continuity') or {}).get('previousFrameUrl')
if previous: refs['previous_frame']=download(resolve_url(previous),job_dir/'previous-frame.png')
motion=job.get('motion') or {}
for key,name in [('poseVideoUrl','pose-video.mp4'),('faceVideoUrl','face-video.mp4')]:
    if motion.get(key): refs[key]=download(resolve_url(motion[key]),job_dir/name)
print('Loaded',job_id,job.get('sceneFamily'),refs)

In [ ]:
def gpu_info():
    if not torch.cuda.is_available(): return {'available':False}
    p=torch.cuda.get_device_properties(0)
    return {'available':True,'name':p.name,'vram_gb':round(p.total_memory/1024**3,1)}
GPU=gpu_info(); print(GPU)
if not GPU.get('available'): raise RuntimeError('Enable a Kaggle GPU')
requested=(job.get('generation') or {}).get('router','auto')
has_motion_controls='poseVideoUrl' in refs and 'faceVideoUrl' in refs
selected=('wan' if requested=='wan' else 'ltx') if requested!='auto' else ('wan' if has_motion_controls and GPU.get('vram_gb',0)>=24 else 'ltx')
print('Router:',selected)

In [ ]:
def first_character_image(refs):
    path=refs.get('previous_frame') or refs.get('lucas') or refs.get('marion')
    if not path: raise RuntimeError('No canonical image reference available')
    return load_image(path)

def run_ltx(job,refs,out_dir):
    from diffusers import LTXImageToVideoPipeline
    model_id=os.environ.get('MONIA_LTX_MODEL_ID','Lightricks/LTX-Video')
    dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    pipe=LTXImageToVideoPipeline.from_pretrained(model_id,torch_dtype=dtype)
    try: pipe.enable_model_cpu_offload()
    except Exception: pipe.to('cuda')
    g=job.get('generation') or {}
    width=int(g.get('width',480)); height=int(g.get('height',832))
    frames=int(g.get('frames',97)); steps=int(g.get('steps',30))
    image=first_character_image(refs)
    video=pipe(image=image,prompt=job['prompt'],negative_prompt=job.get('negativePrompt','identity drift, morphing, distorted face, extra limbs, text, subtitles, watermark, UI'),width=width,height=height,num_frames=frames,num_inference_steps=steps).frames[0]
    path=str(Path(out_dir)/'shot-01.mp4'); export_to_video(video,path,fps=int(g.get('fps',24))); return [path]

def run_wan(job,refs,out_dir):
    if 'poseVideoUrl' not in refs or 'faceVideoUrl' not in refs:
        raise RuntimeError('Wan Animate requires preprocessed poseVideoUrl + faceVideoUrl; raw motion video is not used as identity input')
    from diffusers import WanAnimatePipeline
    model_id=os.environ.get('MONIA_WAN_MODEL_ID','Wan-AI/Wan2.2-Animate-14B-Diffusers')
    dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    pipe=WanAnimatePipeline.from_pretrained(model_id,torch_dtype=dtype)
    pipe.vae.to(torch.float32)
    try: pipe.enable_model_cpu_offload()
    except Exception: pipe.to('cuda')
    character=first_character_image(refs)
    pose=load_video(refs['poseVideoUrl']); face=load_video(refs['faceVideoUrl'])
    out=pipe(image=character,pose_video=pose,face_video=face,prompt=job['prompt']).frames[0]
    path=str(Path(out_dir)/'shot-01.mp4'); export_to_video(out,path,fps=int((job.get('generation') or {}).get('fps',16))); return [path]


In [ ]:
started=time.time(); state='candidate'; errors=[]; clips=[]
try:
    clips=run_wan(job,refs,job_dir) if selected=='wan' else run_ltx(job,refs,job_dir)
except Exception as e:
    state='failed'; errors=[str(e)]
result={'jobId':job_id,'state':state,'candidateOnly':True,'narrativeAuthority':False,'router':selected,'gpu':GPU,'clips':[Path(p).name for p in clips],'errors':errors,'createdAt':time.time(),'elapsedSec':round(time.time()-started,2),'review':{'identityMarion':'pending','identityLucas':'pending','canon':'pending','motion':'pending','continuity':'pending','voice':'pending','wardrobe':'pending','location':'pending'}}
(job_dir/'result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2),encoding='utf-8')
print(json.dumps(result,ensure_ascii=False,indent=2))

## Candidate output only
The folder in `/kaggle/working/monia-studio/<job-id>` is a candidate package. This notebook must never edit `drama-approved.json` or any live manifest. A one-time GitHub/Kaggle connection can later upload candidate folders automatically for review.